In [11]:
import os
import sys
import time
import json
import logging
import pathlib
import urllib.parse
from datetime import datetime, timezone
from typing import Optional

import requests
import pandas as pd

In [12]:
# ============ CONFIG ============
CSV_IN        = "cryptomasun_decoded.csv"              # CSV columns: url,user,timestamp
OUT_DIR       = "Batch_Downloads/Cryptomason"            # where to save videos
API_BASE      = "http://localhost:3000"     # where the BOTCAHX API is running
API_ENDPOINT  = "/tiktok/api.php"           # endpoint path
DRY_RUN       = False                       # True => print actions, don't download
MAX_RETRIES   = 3
RETRY_BACKOFF = 1.6                         # seconds multiplier (exponential backoff)
TIMEOUT       = 60                          # per HTTP request (seconds)
CHUNK_SIZE    = 1024 * 512                  # 512KB stream chunks
# =================================

In [ ]:
# df = pd.read_csv(CSV_IN)
# df.info()
# df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 538 entries, 0 to 537
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   url        538 non-null    object
 1   user       538 non-null    object
 2   timestamp  535 non-null    object
dtypes: object(3)
memory usage: 12.7+ KB


,url,user,timestamp
0,https://www.tiktok.com/@cryptomasun/video/7571...,cryptomasun,2025-11-11T15:13:26Z
1,https://www.tiktok.com/@cryptomasun/video/7570...,cryptomasun,2025-11-09T13:29:27Z
2,https://www.tiktok.com/@cryptomasun/video/7570...,cryptomasun,2025-11-07T18:03:24Z
3,https://www.tiktok.com/@cryptomasun/video/7569...,cryptomasun,2025-11-06T21:29:45Z
4,https://www.tiktok.com/@cryptomasun/video/7569...,cryptomasun,2025-11-06T17:29:58Z


In [13]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
)

# ---------- Helpers ----------

def ensure_dir(p: str) -> None:
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)

def parse_timestamp(ts: str) -> Optional[datetime]:
    """Parse common UTC formats and return tz-aware datetime."""
    if not ts:
        return None
    s = ts.strip()
    try:
        if s.endswith("Z"):
            return datetime.fromisoformat(s.replace("Z", "+00:00"))
        if s.endswith(" UTC"):
            base = datetime.strptime(s[:-4], "%Y-%m-%d %H:%M:%S")
            return base.replace(tzinfo=timezone.utc)
        if len(s) == 10:
            return datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        dt = datetime.fromisoformat(s)
        return dt if dt.tzinfo else dt.replace(tzinfo=timezone.utc)
    except Exception:
        return None

def clean_csv(infile: str) -> pd.DataFrame:
    """Load and clean CSV: strip whitespace, drop invalid rows, dedupe URLs, parse timestamp."""
    df = pd.read_csv(infile)
    df = df.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
    df = df.dropna(subset=["url", "timestamp"])
    df["timestamp_parsed"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    df = df[df["timestamp_parsed"].notna()]
    df = df.drop_duplicates(subset=["url"])
    df = df.sort_values("timestamp_parsed").reset_index(drop=True)
    logging.info("Cleaned CSV: %d valid rows remain.", len(df))
    return df

def is_direct_media(url: str) -> bool:
    """True if the URL looks like a direct mp4 (e.g., tikwm media)."""
    u = url.lower()
    return u.endswith(".mp4") or (".mp4?" in u)

def api_get_video_link(tiktok_url: str) -> Optional[str]:
    """
    Call BOTCAHX API to resolve a video URL.
    Example JSON:
      {"audio":["...mp3"],"video":["https://www.tikwm.com/video/media/play/<id>.mp4"]}
    We return the first element of 'video'.
    """
    api_url = urllib.parse.urljoin(API_BASE, API_ENDPOINT)
    params = {"url": tiktok_url}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(api_url, params=params, timeout=TIMEOUT)
            r.raise_for_status()
            data = r.json()

            # Prefer top-level "video": [...]
            vids = None
            if isinstance(data, dict):
                vids = data.get("video") or data.get("videos")
                if not vids and "result" in data and isinstance(data["result"], dict):
                    vids = data["result"].get("video")
            if isinstance(vids, list) and vids:
                return vids[0]

            # Fallback: scan JSON for an mp4 URL
            blob = json.dumps(data)
            for token in blob.split('"'):
                if token.startswith("http") and ".mp4" in token:
                    return token

            raise RuntimeError("No video link in API response")
        except Exception as e:
            if attempt == MAX_RETRIES:
                logging.error("API failed for %s: %s", tiktok_url, e)
                return None
            delay = RETRY_BACKOFF ** attempt
            logging.warning("API error (attempt %d/%d): %s — retrying in %.1fs",
                            attempt, MAX_RETRIES, e, delay)
            time.sleep(delay)
    return None

def stream_download(src_url: str, dst_path: str) -> bool:
    """Stream the file to disk with retries and atomic rename."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with requests.get(src_url, stream=True, timeout=TIMEOUT, allow_redirects=True) as r:
                r.raise_for_status()
                tmp = dst_path + ".part"
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp, dst_path)
            return True
        except Exception as e:
            if attempt == MAX_RETRIES:
                logging.error("Download failed for %s: %s", src_url, e)
                return False
            delay = RETRY_BACKOFF ** attempt
            logging.warning("Download error (attempt %d/%d): %s — retrying in %.1fs",
                            attempt, MAX_RETRIES, e, delay)
            time.sleep(delay)
    return False

def build_filename(dt: datetime, user: str) -> str:
    """YYYY_MM_DD_user.mp4 with a filesystem-safe username."""
    date_tag = dt.strftime("%Y_%m_%d")
    safe_user = "".join(c for c in (user or "").strip() if c.isalnum() or c in ("-", "_", ".")).strip("_")
    if not safe_user:
        safe_user = "unknown"
    return f"{date_tag}_{safe_user}.mp4"

# ---------- Main ----------

def main():
    ensure_dir(OUT_DIR)

    if not os.path.exists(CSV_IN):
        logging.error("CSV not found: %s", CSV_IN)
        sys.exit(1)

    df = clean_csv(CSV_IN)

    # Keep only year 2025
    df_2025 = df[df["timestamp_parsed"].dt.year == 2025]
    if df_2025.empty:
        logging.info("No rows from 2025 found. Nothing to download.")
        return

    logging.info("Found %d videos from 2025. Starting downloads…", len(df_2025))

    for idx, row in enumerate(df_2025.itertuples(index=False), start=1):
        url = row.url
        user = row.user
        ts   = row.timestamp_parsed

        out_name = build_filename(ts, user)
        out_path = os.path.join(OUT_DIR, out_name)

        if os.path.exists(out_path):
            logging.info("[%d/%d] Skip (exists): %s", idx, len(df_2025), out_name)
            continue

        # If CSV already contains a direct mp4, use it; else call API to resolve
        if is_direct_media(url):
            video_link = url
        else:
            logging.info("[%d/%d] Resolving via API: %s", idx, len(df_2025), url)
            video_link = api_get_video_link(url)

        if not video_link:
            logging.error("[%d/%d] Could not resolve video link. Skipped.", idx, len(df_2025))
            continue

        logging.info("→ %s", out_name)
        if DRY_RUN:
            continue

        ok = stream_download(video_link, out_path)
        if ok:
            logging.info("✓ Saved: %s", out_name)
        else:
            logging.error("✗ Failed: %s", out_name)

    logging.info("All done.")

if __name__ == "__main__":
    main()

22:29:50 INFO: Cleaned CSV: 535 valid rows remain.
22:29:50 INFO: Found 146 videos from 2025. Starting downloads…
22:29:50 INFO: [1/146] Resolving via API: https://www.tiktok.com/@cryptomasun/video/7454973981393751302
22:29:51 INFO: → 2025_01_01_cryptomasun.mp4
22:29:52 INFO: ✓ Saved: 2025_01_01_cryptomasun.mp4
22:29:52 INFO: [2/146] Resolving via API: https://www.tiktok.com/@cryptomasun/video/7455399092743638278
22:29:53 INFO: → 2025_01_02_cryptomasun.mp4
22:29:55 INFO: ✓ Saved: 2025_01_02_cryptomasun.mp4
22:29:55 INFO: [3/146] Resolving via API: https://www.tiktok.com/@cryptomasun/video/7455730485952597253
22:29:56 INFO: → 2025_01_03_cryptomasun.mp4
22:29:56 INFO: ✓ Saved: 2025_01_03_cryptomasun.mp4
22:29:56 INFO: [4/146] Skip (exists): 2025_01_03_cryptomasun.mp4
22:29:56 INFO: [5/146] Resolving via API: https://www.tiktok.com/@cryptomasun/video/7456228434142891269
22:29:57 INFO: → 2025_01_05_cryptomasun.mp4
22:29:57 INFO: ✓ Saved: 2025_01_05_cryptomasun.mp4
22:29:57 INFO: [6/146] Re

### Compress Video Quality

In [7]:
import os
import subprocess
from pathlib import Path

# ============ CONFIG ============
INPUT_DIR  = "Batch_Downloads/Combined"         # your existing folder with downloaded .mp4 files
OUTPUT_DIR = "Batch_Downloads/Combined/compressed"   # compressed output folder
CRF        = 30                       # quality level: 23=default, 28~32 smaller/lower quality
HEIGHT     = 480                      # target video height (keep aspect ratio)
# =================================

In [8]:
def compress_video(in_path: Path, out_path: Path):
    """
    Compress a video to lower resolution and bitrate using ffmpeg.
    """
    cmd = [
        "ffmpeg", "-y", "-i", str(in_path),
        "-vf", f"scale=-2:{HEIGHT}",           # maintain aspect ratio
        "-c:v", "libx264", "-preset", "veryfast",
        "-crf", str(CRF),
        "-c:a", "aac", "-b:a", "96k",
        str(out_path)
    ]
    print(f"🎬 Compressing: {in_path.name} -> {out_path.name}")
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    print(f"✅ Done: {out_path.name}")

def main():
    in_dir = Path(INPUT_DIR)
    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    videos = sorted(in_dir.glob("*.mp4"))
    if not videos:
        print(f"No .mp4 files found in {in_dir}")
        return

    for vid in videos:
        out_path = out_dir / vid.name
        if out_path.exists():
            print(f"⏩ Skipping (already compressed): {vid.name}")
            continue

        try:
            compress_video(vid, out_path)
        except subprocess.CalledProcessError:
            print(f"❌ Compression failed for {vid.name}")

    print("\nAll videos processed.")

if __name__ == "__main__":
    main()

🎬 Compressing: 2025_01_01_coin_guide.mp4 -> 2025_01_01_coin_guide.mp4
✅ Done: 2025_01_01_coin_guide.mp4
🎬 Compressing: 2025_01_01_coin_guide_1.mp4 -> 2025_01_01_coin_guide_1.mp4
✅ Done: 2025_01_01_coin_guide_1.mp4
🎬 Compressing: 2025_01_02_coin_guide.mp4 -> 2025_01_02_coin_guide.mp4
✅ Done: 2025_01_02_coin_guide.mp4
🎬 Compressing: 2025_01_02_coin_guide_1.mp4 -> 2025_01_02_coin_guide_1.mp4
✅ Done: 2025_01_02_coin_guide_1.mp4
🎬 Compressing: 2025_01_02_coin_guide_2.mp4 -> 2025_01_02_coin_guide_2.mp4
✅ Done: 2025_01_02_coin_guide_2.mp4
🎬 Compressing: 2025_01_02_coin_guide_3.mp4 -> 2025_01_02_coin_guide_3.mp4
✅ Done: 2025_01_02_coin_guide_3.mp4
🎬 Compressing: 2025_01_02_coinbureau.mp4 -> 2025_01_02_coinbureau.mp4
✅ Done: 2025_01_02_coinbureau.mp4
🎬 Compressing: 2025_01_02_titovlogs77.mp4 -> 2025_01_02_titovlogs77.mp4
✅ Done: 2025_01_02_titovlogs77.mp4
🎬 Compressing: 2025_01_02_titovlogs77_1.mp4 -> 2025_01_02_titovlogs77_1.mp4
✅ Done: 2025_01_02_titovlogs77_1.mp4
🎬 Compressing: 2025_01_02_ti

### Download Mass Videos 

In [1]:
import os
import sys
import time
import json
import logging
import pathlib
import urllib.parse
import subprocess
from datetime import datetime, timezone
from typing import Optional

import requests
import pandas as pd


In [3]:
# ============ CONFIG ============
CSV_IN        = "combined_decoded.csv"        # CSV columns: url,user,timestamp
OUT_DIR       = "Batch_Downloads/Combined"    # where to save videos

API_BASE      = "http://localhost:3000"          # where the BOTCAHX API is running
API_ENDPOINT  = "/tiktok/api.php"                # endpoint path

DRY_RUN       = False                            # True => print actions, don't download

MAX_RETRIES   = 3
RETRY_BACKOFF = 1.6                              # seconds multiplier (exponential backoff)
TIMEOUT       = 60                               # per HTTP request (seconds)
CHUNK_SIZE    = 1024 * 512                       # 512KB stream chunks

# Throttling to avoid hammering the API / remote site
API_COOLDOWN_SECONDS      = 2.0                  # wait before each API call
DOWNLOAD_COOLDOWN_SECONDS = 1.0                  # wait between downloads
# =================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
)

# ---------- Helpers ----------

def ensure_dir(p: str) -> None:
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)

def parse_timestamp(ts: str) -> Optional[datetime]:
    """Parse common UTC formats and return tz-aware datetime."""
    if not ts:
        return None
    s = ts.strip()
    try:
        if s.endswith("Z"):
            return datetime.fromisoformat(s.replace("Z", "+00:00"))
        if s.endswith(" UTC"):
            base = datetime.strptime(s[:-4], "%Y-%m-%d %H:%M:%S")
            return base.replace(tzinfo=timezone.utc)
        if len(s) == 10:
            return datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        dt = datetime.fromisoformat(s)
        return dt if dt.tzinfo else dt.replace(tzinfo=timezone.utc)
    except Exception:
        return None

def clean_csv(infile: str) -> pd.DataFrame:
    """Load and clean CSV: strip whitespace, drop invalid rows, dedupe URLs, parse timestamp."""
    df = pd.read_csv(infile)
    df = df.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
    df = df.dropna(subset=["url", "timestamp"])
    df["timestamp_parsed"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    df = df[df["timestamp_parsed"].notna()]
    df = df.drop_duplicates(subset=["url"])
    df = df.sort_values("timestamp_parsed").reset_index(drop=True)
    logging.info("Cleaned CSV: %d valid rows remain.", len(df))
    return df

def is_direct_media(url: str) -> bool:
    """True if the URL looks like a direct mp4 (e.g., tikwm media)."""
    u = url.lower()
    return u.endswith(".mp4") or (".mp4?" in u)

def api_get_video_link(tiktok_url: str) -> Optional[str]:
    """
    Call BOTCAHX API to resolve a video URL.
    Expected JSON example:
      {"audio":["...mp3"],"video":["https://www.tikwm.com/video/media/play/<id>.mp4"]}
    We return the first element of 'video'.
    """
    api_url = urllib.parse.urljoin(API_BASE, API_ENDPOINT)
    params = {"url": tiktok_url}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            # Throttle between API calls to reduce 500s / rate-limiting
            time.sleep(API_COOLDOWN_SECONDS)

            r = requests.get(api_url, params=params, timeout=TIMEOUT)
            r.raise_for_status()
            data = r.json()

            vids = None
            if isinstance(data, dict):
                vids = data.get("video") or data.get("videos")
                if not vids and "result" in data and isinstance(data["result"], dict):
                    vids = data["result"].get("video")
            if isinstance(vids, list) and vids:
                return vids[0]

            # Fallback: scan JSON for an mp4 URL
            blob = json.dumps(data)
            for token in blob.split('"'):
                if token.startswith("http") and ".mp4" in token:
                    return token

            raise RuntimeError("No video link in API response")
        except Exception as e:
            if attempt == MAX_RETRIES:
                logging.error("API failed for %s: %s", tiktok_url, e)
                return None
            delay = RETRY_BACKOFF ** attempt
            logging.warning(
                "API error (attempt %d/%d): %s — retrying in %.1fs",
                attempt, MAX_RETRIES, e, delay
            )
            time.sleep(delay)
    return None

def stream_download(src_url: str, dst_path: str) -> bool:
    """Stream the file to disk with retries and atomic rename."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with requests.get(src_url, stream=True, timeout=TIMEOUT, allow_redirects=True) as r:
                r.raise_for_status()
                tmp = dst_path + ".part"
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp, dst_path)
            return True
        except Exception as e:
            if attempt == MAX_RETRIES:
                logging.error("Download failed for %s: %s", src_url, e)
                return False
            delay = RETRY_BACKOFF ** attempt
            logging.warning(
                "Download error (attempt %d/%d): %s — retrying in %.1fs",
                attempt, MAX_RETRIES, e, delay
            )
            time.sleep(delay)
    return False

def build_unique_filename(dt: datetime, user: str, out_dir: str) -> str:
    """
    Build a filesystem-safe filename of the form:
      YYYY_MM_DD_username.mp4
      YYYY_MM_DD_username_1.mp4
      YYYY_MM_DD_username_2.mp4
    ensuring no collisions in out_dir.
    """
    date_tag = dt.strftime("%Y_%m_%d")
    safe_user = "".join(
        c for c in (user or "").strip()
        if c.isalnum() or c in ("-", "_", ".")
    ).strip("_")
    if not safe_user:
        safe_user = "unknown"

    base = f"{date_tag}_{safe_user}"
    candidate = f"{base}.mp4"
    k = 1
    # Bump suffix until we find a free name
    while os.path.exists(os.path.join(out_dir, candidate)):
        candidate = f"{base}_{k}.mp4"
        k += 1
    return candidate

# ---------- Main: DOWNLOAD ONLY ----------

def main():
    ensure_dir(OUT_DIR)

    if not os.path.exists(CSV_IN):
        logging.error("CSV not found: %s", CSV_IN)
        sys.exit(1)

    df = clean_csv(CSV_IN)

    # Only keep year 2025
    df_2025 = df[df["timestamp_parsed"].dt.year == 2025]
    if df_2025.empty:
        logging.info("No rows from 2025 found. Nothing to download.")
        return

    logging.info("Found %d videos from 2025. Starting downloads…", len(df_2025))

    total = len(df_2025)
    for idx, row in enumerate(df_2025.itertuples(index=False), start=1):
        url = row.url
        user = getattr(row, "user", "")
        ts   = row.timestamp_parsed

        out_name = build_unique_filename(ts, user, OUT_DIR)
        out_path = os.path.join(OUT_DIR, out_name)

        if os.path.exists(out_path):
            logging.info("[%d/%d] Skip (exists): %s", idx, total, out_name)
            continue

        # Small delay between downloads generally (extra politeness)
        time.sleep(DOWNLOAD_COOLDOWN_SECONDS)

        # If CSV already contains a direct mp4, use it; else call API to resolve
        if is_direct_media(url):
            video_link = url
        else:
            logging.info("[%d/%d] Resolving via API: %s", idx, total, url)
            video_link = api_get_video_link(url)

        if not video_link:
            logging.error("[%d/%d] Could not resolve video link. Skipped.", idx, total)
            continue

        logging.info("→ %s", out_name)
        if DRY_RUN:
            continue

        ok = stream_download(video_link, out_path)
        if ok:
            logging.info("✓ Saved: %s", out_name)
        else:
            logging.error("✗ Failed: %s", out_name)

    logging.info("Downloads done. ✅ (No compression performed in this script.)")

if __name__ == "__main__":
    main()

00:40:43 INFO: Cleaned CSV: 9256 valid rows remain.
00:40:43 INFO: Found 3206 videos from 2025. Starting downloads…
00:40:44 INFO: [1/3206] Resolving via API: https://www.tiktok.com/@coin_guide/video/7455038690616708383
00:40:47 INFO: → 2025_01_01_coin_guide.mp4
00:40:51 INFO: ✓ Saved: 2025_01_01_coin_guide.mp4
00:40:52 INFO: [2/3206] Resolving via API: https://www.tiktok.com/@coin_guide/video/7455041636645473567
00:40:55 INFO: → 2025_01_01_coin_guide_1.mp4
00:40:58 INFO: ✓ Saved: 2025_01_01_coin_guide_1.mp4
00:41:00 INFO: [3/3206] Resolving via API: https://www.tiktok.com/@titovlogs77/video/7455121144543251719
00:41:02 INFO: → 2025_01_02_titovlogs77.mp4
00:41:04 INFO: ✓ Saved: 2025_01_02_titovlogs77.mp4
00:41:05 INFO: [4/3206] Resolving via API: https://www.tiktok.com/@titovlogs77/video/7455167006157868308
00:41:08 INFO: → 2025_01_02_titovlogs77_1.mp4
00:41:10 INFO: ✓ Saved: 2025_01_02_titovlogs77_1.mp4
00:41:11 INFO: [5/3206] Resolving via API: https://www.tiktok.com/@titovlogs77/vid

KeyboardInterrupt: 